[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/heat_cycles/heat_cycles.ipynb)

# The Carnot Cycle

This example computes the heat and work of a Carnot engine. A gas in a cylinder absorbs heat from a hot reservoir while expanding, expands further under thermal insulation until it cools to a cold reservoir's temperature, rejects heat to the cold reservoir while being compressed, and is compressed under insulation back to its starting state. Because the gas's internal energy returns to its initial value, the net work done over the cycle equals the heat absorbed from the hot reservoir minus the heat rejected to the cold reservoir.

A state of the gas is modelled as an antivector in the exterior algebra `s0 v0` of entropy and volume, `entropies * entropy + volumes * volume`, and the gas's internal energy is a scalar function of its state. Temperature and pressure are extrinsic potentials along `s` and `v`, so along a state tangent the heat absorbed is `(temperature * s) & tangent` and the work done is `(pressure * v) & tangent`. Their difference equals the differential of the internal energy, whereas neither one-form alone is the gradient of a state function, so heat and work depend on the path while their difference depends only on the endpoints. Wedging each potential's state gradient with its direction gives the two cycle-area two-forms `temperature_slope ^ s` and `pressure_slope ^ v`, which pair with oriented patches of state changes via `&`; because both potentials come from one internal energy, the two area forms are equal at every state, making the enclosed net-heat area on the temperature–entropy chart equal to the enclosed net-work area on the pressure–volume chart.

In classical thermodynamic integral and partial-derivative notation, net heat and work around a closed cycle are the path integrals $\oint \delta Q = \oint T\,dS$ and $\oint \delta W = \oint p\,dV$ linked by $dU = T\,dS - p\,dV$, and equating the enclosed areas on the $(T, S)$ and $(p, V)$ planes changes variables through the Jacobian determinant $\frac{\partial(T, S)}{\partial(p, V)} = \left(\frac{\partial T}{\partial p}\right)_{\!V}\!\left(\frac{\partial S}{\partial V}\right)_{\!p} - \left(\frac{\partial T}{\partial V}\right)_{\!p}\!\left(\frac{\partial S}{\partial p}\right)_{\!V} = 1$ (written $dT \wedge dS = dp \wedge dV$ in differential-form notation) via $\frac{\partial^2 U}{\partial V\,\partial S} = \frac{\partial^2 U}{\partial S\,\partial V}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext
from examples.thermodynamics.heat_cycles import render, scenarios
from examples.util.animation import show_animation
from numga.topology import Topology

ga = Algebra("s0v0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                 # Scalar+
State = ga.gatype.antivector()                              # Antivector-
Potential = ga.gatype.vector()                              # Vector+
AreaForm = ga.gatype.bivector()                             # Bivector+

# Unit state directions and the unit oriented patch of state space.
# Entropy is in units of nR, volume and temperature in reference values, and energy in units of nRT_0.
entropy = mv.s.dual()                                       # [] State-
volume = mv.v.dual()                                        # [] State-
unit_patch = entropy & volume                               # [] Antibivector-

## The cycle

The cycle connects four corners in entropy and temperature by straight segments. Along the two reservoir strokes the entropy changes at constant temperature, and along the two insulated strokes the temperature changes at constant entropy. At any entropy and temperature, the ideal gas occupies the volume where its internal energy's slope along entropy equals that temperature.

In [ ]:
# Heat capacity, reservoir temperatures, entropy span, and step count per stroke.
capacity = scenarios.capacity
hot, cold, entropy_span = scenarios.hot, scenarios.cold, scenarios.entropy_span
n_steps = scenarios.n_steps
corner_entropy = mv.scalar([[0], [entropy_span], [entropy_span], [0]]).field()       # Scalar+[V]
corner_temperature = mv.scalar([[hot], [hot], [cold], [cold]]).field()               # Scalar+[V]
around = Topology.loops([corner_entropy.sites.count])

def energy(state: State) -> Scalar:
    """Internal energy of an ideal gas with constant heat capacity at fixed volume."""
    return capacity * (((mv.s & state) - (mv.v & state).log()) / capacity).exp()

def temperature_of(state: State) -> Scalar:
    """Absolute temperature of the ideal gas at each state."""
    return energy(state) / capacity

def pressure_of(state: State) -> Scalar:
    """Pressure of the ideal gas at each state."""
    return temperature_of(state) / (mv.v & state)

def equilibrium_states(entropies: Scalar, temperatures: Scalar) -> State:
    """Equilibrium states of the ideal gas at the given entropies and temperatures."""
    return entropy * entropies + volume * (entropies - capacity * temperatures.log()).exp()

# Gather each stroke's starting corner and corner difference onto the cycle's steps.
by_stroke = around.reading(np.repeat(np.arange(around.n_vertices), n_steps), 1)      # Scalar+[V_samples <- E]
entropy_starts = by_stroke * (around.incidence[1, 0][0] * corner_entropy)            # Scalar+[V_samples]
entropy_steps = by_stroke * (around.d[0] * corner_entropy)                           # Scalar+[V_samples]
temperature_starts = by_stroke * (around.incidence[1, 0][0] * corner_temperature)    # Scalar+[V_samples]
temperature_steps = by_stroke * (around.d[0] * corner_temperature)                   # Scalar+[V_samples]

def stroke_samples(fractions: Scalar) -> tuple[State, State]:
    """States and stroke tangents at the given step fractions around the closed loop of corners."""
    entropies = entropy_starts + entropy_steps * fractions                           # Scalar+[V_samples]
    temperatures = temperature_starts + temperature_steps * fractions                # Scalar+[V_samples]
    states = equilibrium_states(entropies, temperatures)                             # State-[V_samples]
    # Differentiating V = exp(S - c ln T) along the linear (S, T) stroke gives dV = V * (dS - c dT / T).
    volumes = mv.v & states                                                          # Scalar+[V_samples]
    tangents = entropy * entropy_steps + volume * volumes * (entropy_steps - capacity * temperature_steps / temperatures)
    return states, tangents

starts = mv.scalar(np.tile(np.arange(n_steps) / n_steps, around.n_vertices)[:, None]).field()  # Scalar+[V_samples]
cycle, _ = stroke_samples(starts)                                                    # State-[V_samples]
render.draw_cycle(temperature_of, pressure_of, cycle);

The pressure–volume chart on the left shows the cycle against red isotherms and teal adiabats, and the temperature–entropy chart on the right shows it against green isochores and purple isobars.

## Heat and work along the strokes

At each state, pairing the heat potential `temperature * s` and the work potential `pressure * v` with the stroke tangent gives the rates of heat absorption and work along the stroke. Summing those midpoint rates back onto the four strokes gives each stroke's total heat and work. Heat enters and leaves only along the two isothermal strokes, whereas mechanical work occurs on all four strokes, and on every stroke their difference equals the change in internal energy.

In [ ]:
def exchanges(states: State, tangents: State) -> tuple[Scalar, Scalar]:
    """Heat absorbed and work done along each state tangent."""
    heat_form = temperature_of(states) * mv.s                                        # [...] Potential+
    work_form = pressure_of(states) * mv.v                                           # [...] Potential+
    return heat_form & tangents, work_form & tangents

middles = mv.scalar(np.tile((np.arange(n_steps) + 0.5) / n_steps, around.n_vertices)[:, None]).field()  # Scalar+[V_samples]
mid_states, tangents = stroke_samples(middles)                                       # State-[V_samples], State-[V_samples]
heat_rate, work_rate = exchanges(mid_states, tangents)                               # Scalar+[V_samples], Scalar+[V_samples]
# Sum the midpoint step contributions back onto their strokes via the transpose of the stroke reading.
stroke_heat = by_stroke.adjugate() * (heat_rate / n_steps)                                     # Scalar+[E]
stroke_work = by_stroke.adjugate() * (work_rate / n_steps)                                     # Scalar+[E]

In [ ]:
print("heat absorbed by stroke:", stroke_heat.batch().to_array())
print("work done by stroke:    ", stroke_work.batch().to_array())
print("efficiency:", (stroke_work.sites.sum() / stroke_heat.batch()[0]).to_array())

## Equal areas

Wedging the state gradient of temperature with `s` gives the heat area two-form `temperature_slope ^ s`, and wedging the state gradient of pressure with `v` gives the work area two-form `pressure_slope ^ v`. Because both potentials are slopes of one internal energy, the mixed terms match and the two bivectors are identical at every state, so the enclosed areas marked W on the two charts are equal around any closed cycle.

In [ ]:
def potential_slopes(state: State) -> tuple[Potential, Potential]:
    """State gradients of temperature and pressure across entropy and volume."""
    temperatures = temperature_of(state)                                             # [...] Scalar+
    volumes = mv.v & state                                                           # [...] Scalar+
    pressures = temperatures / volumes                                               # [...] Scalar+
    temperature_slope = (temperatures / capacity) * (mv.s - mv.v / volumes)          # [...] Potential+
    pressure_slope = (pressures / capacity) * mv.s - (pressures * (1 + 1 / capacity) / volumes) * mv.v  # [...] Potential+
    return temperature_slope, pressure_slope

def circulations(state: State) -> tuple[AreaForm, AreaForm]:
    """Heat and work two-forms circulated per oriented patch of state space."""
    temperature_slope, pressure_slope = potential_slopes(state)                      # [...] Potential+, Potential+
    return temperature_slope ^ mv.s, pressure_slope ^ mv.v

heat_area, work_area = circulations(cycle)                                           # AreaForm+[V_samples], AreaForm+[V_samples]

In [ ]:
print("heat circulated per unit patch at the corners:", (heat_area & unit_patch).batch()[::n_steps].to_array())
print("work circulated per unit patch at the corners:", (work_area & unit_patch).batch()[::n_steps].to_array())

## Running the engine

Stepping around the cycle raises the piston during isothermal heating and insulated expansion, and lowers it during isothermal cooling and insulated compression.

In [ ]:
def cycle_states() -> Iterator[State]:
    """Yield each gas state in turn around the cycle."""
    for state in cycle.batch():
        yield state

show_animation(render.animate_cycle(temperature_of, pressure_of, cycle, cycle_states()),
               "thermodynamics_heat_cycle", duration_ms=scenarios.frame_duration_ms)

Orange marks heat absorbed from the hot reservoir, blue marks heat rejected to the cold reservoir, and grey marks the two insulated strokes. Over one cycle the gas absorbs 1.5 units of heat, rejects 1 unit, and performs 0.5 units of net work, matching the Carnot efficiency `1 - cold / hot` of one third.

In [ ]:
# checks
# Differences from each cycle state to the next sum to zero around the closed loop.
np.testing.assert_allclose((Topology.loops([cycle.sites.count]).d[0] * cycle).sites.sum().kernel, 0, atol=1e-10)
np.testing.assert_allclose((stroke_heat.batch() - mv.scalar([[hot * entropy_span], [0], [-cold * entropy_span], [0]])).kernel,
                           0, atol=1e-10)
np.testing.assert_allclose((stroke_work.sites.sum() - (hot - cold) * entropy_span).kernel, 0, atol=1e-10)
# Along each stroke, heat absorbed minus work done equals the change in internal energy between its corners.
energy_change = around.d[0] * energy(equilibrium_states(corner_entropy, corner_temperature))   # Scalar+[E]
np.testing.assert_allclose((stroke_heat - stroke_work - energy_change).kernel, 0, atol=1e-10)
np.testing.assert_allclose((heat_area - work_area).kernel, 0, atol=1e-10)